In [1]:
# Test Environment
import ee
import geemap
import pygadm
import geopandas as gpd
import geojson

In [2]:
# Connect to GEE
ee.Initialize(project="tutorial-geoscripting-paulian")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [3]:
# Import image
s1_collection = ee.ImageCollection("COPERNICUS/S1_GRD")

In [4]:
# set our initial zoom position for Constanta, Romania
center_lat = 44.1092
center_lon = 28.7499
zoomlevel = 12

# The following line will initialize the map
Map = geemap.Map(center=[center_lat, center_lon], zoom=zoomlevel)

# The following line will actually call and show the map:
Map

Map(center=[44.1092, 28.7499], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
# Small time interval to get 1 snapshot
from_date = '2025-10-15'
to_date = '2025-10-31'

# Constanța port plus the offshore anchorage zone
bbox = [28.64, 44.08, 28.85, 44.18]
aoi = ee.Geometry.Rectangle(bbox)

# Image filter
image = s1_collection\
            .filterBounds(aoi)\
            .filterDate(from_date,to_date)\
            .sort("system:time_start", True)\
            .first()

# VV Polarization
vizualisation_params = {
    'min': -30,
    'max': -5,
    'opacity':1,
    'bands':['VH']
}

# Add image to base map
Map.addLayer(ee_object = image.select('VH').clip(aoi),
             vis_params = vizualisation_params,
             name = "first image")

In [ ]:
from functions.detection_algo import load_water
water = load_water('images/port_osm/clip_water.shp', bbox) #shapefile source: "https://osmdata.openstreetmap.de/data/water-polygons.html"

In [8]:
from functions.detection_algo import detect_ships

ships = detect_ships(image, water)
print(ships.size().getInfo())                   # vessel count
Map.addLayer(water, {'color': 'blue'}, 'water mask')
Map.addLayer(ships, {'color': 'red'}, 'ships')

176


In [ ]:
print(image.date().format('YYYY-MM-dd HH:mm').getInfo())